# Do the two axle configurations add information together?
Jamie Le · November 11, 2026

**Question:** Does keeping an unordered pair of axle descriptors improve truck hunting prediction beyond the existing means/maxima and standard wear?

Notebook 05 tests separate axle predictions followed by OR, which changes the training labels and doubles the row count. We also need to check whether direct truck training can preserve both axles. Here the original truck labels and number of observations are held fixed. The model learns directly from the two axle vectors together.

We test the profile representation, using the same kernel-ridge family. It complements the existing wear/conicity, rail geometry and deep-learning work. The three shape summaries are retained; full raw wheel curves, front/rear roles and signed left/right arrangements are not modeled.

## Reproduction

We use this notebook for our profile-model comparisons in the report and presentation. Install Python 3.12 and the packages in [requirements.txt](../requirements.txt), following [INSTALL.md](../INSTALL.md). We need only the base packages; this notebook does not require TensorFlow, XGBoost or MATLAB.

Get the data through the private **-MxV-Wheel-Profile** course Teams channel: **Shared → Documents → Data**. Download `Profiles/`, `Eq_Con_Fit_Results.xlsm` and `final_merged_data.csv`, preserving the original folder structure. These profile experiments read:

```text
data/
├── final_merged_data.csv
└── Profiles/
    ├── Candidate wheel profiles/Car*_Axle*[LR].WHL
    └── WNTs/2023-06-21 - Hunt_Disp_Max_CP_Difference.csv
```

We expect all 112 measured WHL files, separate from the unworn reference. We also accept `final_merged_data(in).csv`. If you keep the downloaded folder outside the repository, set its location before starting Jupyter:

macOS/Linux:
```bash
export HUNTING_DATA_DIR="/full/path/to/Data"
```

Windows PowerShell:
```powershell
$env:HUNTING_DATA_DIR = "C:\full\path\to\Data"
```

The default is the repository's `data/` folder. A locally rebuilt `outputs/data/final_merged_data.csv` takes precedence; regenerate or remove it when changing input releases.

From the repository root, activate the environment and start `python -m notebook`. Open this notebook and choose **Restart Kernel and Run All Cells**. We include the experiment functions below, so no unpublished helper modules, previous notebook session or historical-result files are required. The existing `src/contact_hunting.py` and `src/profile_shapes.py` supply the shared kernel and WHL readers.

We export figures as 300 dpi PNGs and vector PDFs and keep the tables behind them under the local output folder named below. The model runs recompute settings and predictions, using complete-car folds and 2,000 bootstrap draws with seed 42. To inspect a completed run without retraining, open its local tables or saved executed notebook. Those results apply to that run's inputs and code; rerun after either changes.

Keep partner-derived tables, images and executed notebooks in the private Teams storage. The Git notebook has cleared outputs. Confirm instructor access to the private data and any saved intermediates before submission.

**Output folder:** `outputs/paired_axles/`.


In [ ]:
from pathlib import Path
import os
import sys
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project_folder = next((p for p in (Path.cwd(), *Path.cwd().parents)
                       if (p / "src" / "contact_hunting.py").is_file()), None)
if project_folder is None:
    raise RuntimeError("Start Jupyter in the repository root or notebooks/; see INSTALL.md.")
sys.path.insert(0, str(project_folder / "src"))
configured_data = Path(os.environ.get("HUNTING_DATA_DIR", "data")).expanduser()
data_folder = configured_data if configured_data.is_absolute() else project_folder / configured_data

def merged_path(root):
    paths = [Path(root) / "outputs/data/final_merged_data.csv",
             data_folder / "final_merged_data.csv", data_folder / "final_merged_data(in).csv"]
    for path in paths:
        if path.is_file():
            return path
    raise FileNotFoundError("Missing merged CSV. See this notebook's input instructions and INSTALL.md.")

folder = project_folder / "outputs" / "paired_axles"
folder.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 120})


## Experiment functions

We define the own-axle descriptors, shared input checks and unordered-pair kernel below. We average both axle matchings, pool scaling within training cars and apply rail similarity once.

In [ ]:
"""Axle-own-geometry experiment with complete-car nested evaluation.

The training label belongs to an axle. Model settings are selected on inner
truck balanced accuracy after a fixed OR rule; native axle performance is a
separate descriptive result. Decision scores are not probabilities.
"""

from pathlib import Path
import sys

import numpy as np
import pandas as pd

from contact_hunting import (
    DISP_SPEED, PARAMETER_GRID, RAIL_PROFILES, SPEED_LIMIT,
    classification_metrics, kernel_scores,
)
from profile_shapes import SHAPE_COLUMNS, TREAD_GRID, describe_tread


TRUCK_KEY = ["Car", "Truck", "Rail_Profile"]
AXLE_KEY = ["Car", "Truck", "Axle", "Rail_Profile"]
WEAR_SOURCES = {
    "wear_flange_width": ["Flange Width_L", "Flange Width_R"],
    "wear_flange_height": ["Flange Height (Sh)_L", "Flange Height (Sh)_R"],
    "wear_abs_hollow": ["|Max Hollow Wear - AAR|_L", "|Max Hollow Wear - AAR|_R"],
    "wear_hollow_position": ["Max Hollow Wear Position - AAR_L", "Max Hollow Wear Position - AAR_R"],
}


def _checked_keys(frame):
    missing = set(AXLE_KEY).difference(frame.columns)
    if missing or frame.empty:
        raise ValueError("Axle assessments need nonempty complete identifiers.")
    ids = frame[AXLE_KEY].to_numpy(dtype=float)
    if not np.isfinite(ids).all() or (ids != np.floor(ids)).any():
        raise ValueError("Car, truck, axle and rail identifiers must be finite integers.")
    if (frame.Car <= 0).any() or not frame.Axle.isin([1, 2, 3, 4]).all():
        raise ValueError("Unsupported measured car or axle identifier.")
    if not np.array_equal(frame.Truck, np.where(frame.Axle <= 2, 1, 2)):
        raise ValueError("Truck identifiers disagree with their axle mapping.")
    if not frame.Rail_Profile.isin(RAIL_PROFILES).all():
        raise ValueError("Unsupported rail identifier.")
    if frame.duplicated(AXLE_KEY).any():
        raise ValueError("Duplicate axle/rail assessments.")
    for (_, truck, _), group in frame.groupby(TRUCK_KEY, sort=True):
        expected = {1, 2} if truck == 1 else {3, 4}
        if len(group) != 2 or set(group.Axle) != expected:
            raise ValueError("Each truck/rail assessment requires its two correct axles.")


def prepare_axles(merged, wheels, profiles):
    """Make own-axle features without reading or modifying source files.

    Wear is averaged over the two sides of the same axle. Source rail rows
    never change the weight assigned to a wheel. Shape definitions use the
    existing centered tread window, whose physical interpretation is provisional.
    """
    _checked_keys(merged)
    sources = [column for pair in WEAR_SOURCES.values() for column in pair]
    needed = ["Eq Con 3mm", DISP_SPEED, *sources]
    if not set(needed).issubset(merged):
        raise ValueError("Required axle conicity, displacement or wear columns are missing.")
    numeric = merged[needed].to_numpy(dtype=float)
    if not np.isfinite(numeric).all():
        raise ValueError("Axle measurements must be finite.")
    if ((merged[DISP_SPEED] <= 0) | (merged[DISP_SPEED] > SPEED_LIMIT)).any():
        raise ValueError("Axle displacement speeds must be in (0, 80].")
    if (merged.groupby(["Car", "Axle"])[sources].nunique() != 1).any().any():
        raise ValueError("Axle wear measurements must be invariant across rail rows.")

    wheel_columns = {"Car", "Axle", "Side", "tread_slope", "tread_nonlinearity"}
    if not wheel_columns.issubset(wheels):
        raise ValueError("Required own-wheel descriptors are missing.")
    if wheels.duplicated(["Car", "Axle", "Side"]).any():
        raise ValueError("Duplicate wheel descriptors.")
    expected = {(int(car), int(axle), side)
                for car, axle in merged[["Car", "Axle"]].drop_duplicates().itertuples(index=False)
                for side in ("L", "R")}
    wheel_keys = set(wheels[["Car", "Axle", "Side"]].itertuples(index=False, name=None))
    if wheel_keys != expected or set(profiles) != expected:
        raise ValueError("Own-wheel and profile coverage must match the measured axles.")
    if not np.isfinite(wheels[["tread_slope", "tread_nonlinearity"]].to_numpy(float)).all():
        raise ValueError("Own-wheel descriptors must be finite.")
    wheel_index = wheels.set_index(["Car", "Axle", "Side"]).sort_index()
    shape_rows = []
    for car, axle in sorted({(c, a) for c, a, _ in expected}):
        for side in ("L", "R"):
            slope, nonlinear = describe_tread(profiles[(car, axle, side)])
            supplied = wheel_index.loc[(car, axle, side)]
            if not np.allclose([slope, nonlinear],
                               [supplied.tread_slope, supplied.tread_nonlinearity],
                               rtol=1e-10, atol=1e-12):
                raise ValueError("Wheel descriptors disagree with their own profile.")
        own = wheel_index.loc[(car, axle)]
        difference = np.asarray(profiles[(car, axle, "L")]) - profiles[(car, axle, "R")]
        shape_rows.append({
            "Car": car, "Axle": axle,
            SHAPE_COLUMNS[0]: own.tread_slope.mean(),
            SHAPE_COLUMNS[1]: own.tread_nonlinearity.mean(),
            # Keep the original name so feature bundles can match. Here there
            # is one axle, so no maximum across the truck's two axles is taken.
            SHAPE_COLUMNS[2]: float(np.sqrt(np.mean(difference ** 2))),
        })
    result = merged[AXLE_KEY + ["Eq Con 3mm", DISP_SPEED] + sources].copy()
    result["conicity_mean"] = result["Eq Con 3mm"]
    result["displacement_hunted"] = (result[DISP_SPEED] < SPEED_LIMIT).astype(int)
    for name, pair in WEAR_SOURCES.items():
        result[name] = result[pair].mean(axis=1)
    retained = AXLE_KEY + ["displacement_hunted", "conicity_mean", *WEAR_SOURCES]
    result = result[retained].merge(pd.DataFrame(shape_rows), on=["Car", "Axle"],
                                     how="left", validate="many_to_one")
    return result.sort_values(AXLE_KEY).reset_index(drop=True)


def aggregate_scores(frame, scores):
    """Return true OR label and fixed max-score OR prediction per truck/rail."""
    _checked_keys(frame)
    values = np.asarray(scores, dtype=float)
    if values.shape != (len(frame),) or not np.isfinite(values).all():
        raise ValueError("One finite signed score is needed per axle assessment.")
    if "displacement_hunted" not in frame or not frame.displacement_hunted.isin([0, 1]).all():
        raise ValueError("Axle displacement labels must be binary.")
    paired = frame[AXLE_KEY + ["displacement_hunted"]].copy()
    paired["score"] = values
    trucks = paired.groupby(TRUCK_KEY, as_index=False, sort=True).agg(
        actual=("displacement_hunted", "max"), score=("score", "max"))
    trucks["predicted"] = (trucks.score >= 0).astype(int)
    return trucks


def evaluate_axle_models(frame, feature_sets, parameter_grid=PARAMETER_GRID):
    """Nested leave-car-out axle training, tuned using pooled inner truck BA.

    Returns wide axle predictions (including raw scores), wide truck predictions
    after OR, and selected settings. These tables contain partner-derived data
    and remain local. The same numeric grid fixes search budget, not identical
    effective regularization to truck training with half as many rows.
    """
    _checked_keys(frame)
    if "displacement_hunted" not in frame or not frame.displacement_hunted.isin([0, 1]).all():
        raise ValueError("Axle displacement labels must be binary.")
    frame = frame.reset_index(drop=True)
    car_ids = frame.Car.to_numpy()
    cars = np.sort(np.unique(car_ids))
    if len(cars) < 4:
        raise ValueError("At least four complete car groups are required.")
    y = frame.displacement_hunted.to_numpy(dtype=int)
    if set(np.unique(y)) != {0, 1}:
        raise ValueError("Both displacement classes are required.")
    allowed = {"conicity_mean", *WEAR_SOURCES, *SHAPE_COLUMNS}
    reserved = set(AXLE_KEY + ["actual", "score", "predicted"])
    if not feature_sets or any(name in reserved or name.endswith("__score") for name in feature_sets):
        raise ValueError("Feature-set names must be nonempty and preserve output columns.")
    if any(not set(columns).issubset(allowed) for columns in feature_sets.values()):
        raise ValueError("Only own-axle geometry and wear predictors are allowed.")
    grid = tuple(parameter_grid)
    if not grid or any(len(pair) != 2 or not np.isfinite(pair).all() or min(pair) <= 0 for pair in grid):
        raise ValueError("Kernel settings must be finite positive alpha/gamma pairs.")
    features = {name: frame[columns].to_numpy(dtype=float) for name, columns in feature_sets.items()}
    if any(not np.isfinite(x).all() for x in features.values()):
        raise ValueError("Model features must be finite.")
    rails = frame.Rail_Profile.to_numpy()
    axle_predictions = frame[AXLE_KEY].copy()
    axle_predictions["actual"] = y
    truck_predictions = aggregate_scores(frame, np.zeros(len(frame)))[TRUCK_KEY + ["actual"]]
    settings = []

    def score_model(name, fit, test, alpha, gamma):
        if set(car_ids[fit]) & set(car_ids[test]):
            raise AssertionError("Training and validation/test cars overlap.")
        if set(np.unique(y[fit])) != {0, 1}:
            raise ValueError("A training fold has only one displacement class.")
        return kernel_scores(features[name][fit], features[name][test],
                             rails[fit], rails[test], y[fit], alpha, gamma)

    for name in feature_sets:
        outer_scores = np.full(len(frame), np.nan)
        seen = np.zeros(len(frame), dtype=int)
        for held_car in cars:
            train = np.flatnonzero(car_ids != held_car)
            test = np.flatnonzero(car_ids == held_car)
            inner_folds = np.array_split(cars[cars != held_car], 3)
            candidates = []
            for alpha, gamma in grid:
                inner_scores = np.full(len(frame), np.nan)
                inner_seen = np.zeros(len(frame), dtype=int)
                for validation_cars in inner_folds:
                    validation = train[np.isin(car_ids[train], validation_cars)]
                    fit = train[~np.isin(car_ids[train], validation_cars)]
                    inner_scores[validation] = score_model(name, fit, validation, alpha, gamma)
                    inner_seen[validation] += 1
                if not np.all(inner_seen[train] == 1) or np.any(inner_seen[test]):
                    raise AssertionError("Inner validation did not cover training cars exactly once.")
                inner_trucks = aggregate_scores(frame.iloc[train], inner_scores[train])
                ba = classification_metrics(inner_trucks.actual, inner_trucks.predicted)["balanced_accuracy"]
                candidates.append((ba, float(alpha), float(gamma)))
            ba, alpha, gamma = max(candidates, key=lambda item: (item[0], item[1], -item[2]))
            outer_scores[test] = score_model(name, train, test, alpha, gamma)
            seen[test] += 1
            settings.append(dict(held_car=int(held_car), model=name, alpha=alpha, gamma=gamma,
                                 inner_truck_balanced_accuracy=ba))
        if not np.all(seen == 1) or not np.isfinite(outer_scores).all():
            raise AssertionError("Each axle assessment must receive one finite held-out score.")
        axle_predictions[name] = (outer_scores >= 0).astype(int)
        axle_predictions[name + "__score"] = outer_scores
        aggregate = aggregate_scores(frame, outer_scores)
        pd.testing.assert_frame_equal(truck_predictions[TRUCK_KEY + ["actual"]],
                                      aggregate[TRUCK_KEY + ["actual"]])
        truck_predictions[name] = aggregate.predicted.to_numpy()
    return axle_predictions, truck_predictions, pd.DataFrame(settings)


In [ ]:
"""Fixed local axle-versus-truck experiment. Source inputs are read only."""
from pathlib import Path
import hashlib
import json
import os
import platform
import sys

import numpy as np
import pandas as pd

ROOT = project_folder
DATA = data_folder
from contact_hunting import (load_tables, prepare_trucks, evaluate_models,
                             classification_metrics, paired_car_intervals)
from profile_shapes import prepare_truck_shapes, SHAPE_COLUMNS

FEATURE_SETS = {
    "Conicity": ["conicity_mean"],
    "Conicity + shapes": ["conicity_mean", *SHAPE_COLUMNS],
    "Wear": ["conicity_mean", *WEAR_SOURCES],
    "Wear + shapes": ["conicity_mean", *WEAR_SOURCES, *SHAPE_COLUMNS],
}
KEY = ["Car", "Truck", "Rail_Profile"]
OLD_NAMES = {
    "Conicity": "Mean conicity + rail",
    "Conicity + shapes": "Mean conicity + shape + rail",
    "Wear": "Wear + conicity + rail",
    "Wear + shapes": "Wear + conicity + all shapes + rail",
}
PREVIOUS = Path(os.environ["HUNTING_REFERENCE_DIR"]) if os.environ.get("HUNTING_REFERENCE_DIR") else None


def hashes(paths, relative_to):
    return {str(p.relative_to(relative_to)) if p.is_relative_to(relative_to) else p.name: hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(paths)}


def protected_state():
    """Record input and tracked-code hashes, excluding generated outputs."""
    sources = [merged_path(ROOT), DATA / "Profiles/WNTs/2023-06-21 - Hunt_Disp_Max_CP_Difference.csv"]
    sources += list((DATA / "Profiles/Candidate wheel profiles").glob("*.WHL"))
    code = list((ROOT / "src").glob("*.py"))
    code += list((ROOT / "notebooks").glob("*.ipynb"))
    code += list((ROOT / "tests").glob("*.py"))
    return {"source_hashes": hashes(sources, ROOT), "code_hashes": hashes(code, ROOT)}


def load_inputs():
    merged = pd.read_csv(merged_path(ROOT))
    contact = pd.read_csv(DATA / "Profiles/WNTs/2023-06-21 - Hunt_Disp_Max_CP_Difference.csv", encoding="cp1252")
    trucks = prepare_trucks(merged, contact)
    shapes, wheels, profiles = prepare_truck_shapes(DATA / "Profiles/Candidate wheel profiles", trucks)
    trucks = trucks.merge(shapes, on=["Car", "Truck"], validate="many_to_one")
    axles = prepare_axles(merged, wheels, profiles)
    wear = axles.drop_duplicates(["Car", "Axle"]).groupby(["Car", "Truck"], as_index=False)[list(WEAR_SOURCES)].mean()
    trucks = trucks.merge(wear, on=["Car", "Truck"], validate="many_to_one")
    if PREVIOUS is not None:
        prior = pd.read_csv(PREVIOUS / "truck_features.csv")
        pd.testing.assert_frame_equal(trucks.sort_values(KEY).reset_index(drop=True),
                                      prior.sort_values(KEY).reset_index(drop=True),
                                      check_exact=False, rtol=1e-12)
    actual = aggregate_scores(axles, axles.displacement_hunted.to_numpy(float) - .5)
    check = trucks.merge(actual, on=KEY, validate="one_to_one")
    assert np.array_equal(check.displacement_hunted, check.actual)
    assert len(axles) == 392 and len(trucks) == 196 and trucks.Car.nunique() == 14
    return trucks, axles, wheels, profiles


In [ ]:
"""Truck-level kernel ridge on unordered pairs of own-axle feature vectors.

Both matchings are averaged, then the numeric pair kernel is normalized to
unit diagonal. Rail similarity enters once. No axle labels or OR predictions
are used in this experiment, and signed scores are not probabilities.
"""

from pathlib import Path
import sys

import numpy as np
import pandas as pd

from contact_hunting import PARAMETER_GRID, RAIL_PROFILES, classification_metrics
from profile_shapes import SHAPE_COLUMNS


TRUCK_KEY = ["Car", "Truck", "Rail_Profile"]
AXLE_KEY = ["Car", "Truck", "Axle", "Rail_Profile"]
WEAR_COLUMNS = ["wear_flange_width", "wear_flange_height", "wear_abs_hollow", "wear_hollow_position"]
ALLOWED_FEATURES = {"conicity_mean", *WEAR_COLUMNS, *SHAPE_COLUMNS}
DISTANCE_SCALE = 9.0


def _check_truck_keys(frame):
    if frame.empty or not set(TRUCK_KEY).issubset(frame):
        raise ValueError("Truck assessments need nonempty complete identifiers.")
    identifiers = frame[TRUCK_KEY].to_numpy(dtype=float)
    if not np.isfinite(identifiers).all() or (identifiers != np.floor(identifiers)).any():
        raise ValueError("Truck keys must be finite integers.")
    if not frame.Car.gt(0).all() or not frame.Truck.isin([1, 2]).all():
        raise ValueError("Unsupported car or truck identifier.")
    if not frame.Rail_Profile.isin(RAIL_PROFILES).all():
        raise ValueError("Unsupported rail identifier.")
    if frame.duplicated(TRUCK_KEY).any():
        raise ValueError("Duplicate truck/rail assessments.")


def make_pair_tensor(trucks, axles, columns):
    """Align both own-axle vectors to each input truck row, preserving row order."""
    _check_truck_keys(trucks)
    if not columns or not set(columns).issubset(ALLOWED_FEATURES):
        raise ValueError("Only prespecified own-axle geometry and wear predictors are allowed.")
    if not set(AXLE_KEY + list(columns)).issubset(axles):
        raise ValueError("Axle keys or own-axle predictors are missing.")
    if axles.empty or axles.duplicated(AXLE_KEY).any():
        raise ValueError("Axle assessments must be nonempty and unique.")
    identifiers = axles[AXLE_KEY].to_numpy(dtype=float)
    if not np.isfinite(identifiers).all() or (identifiers != np.floor(identifiers)).any():
        raise ValueError("Axle keys must be finite integers.")
    if not axles.Axle.isin([1, 2, 3, 4]).all():
        raise ValueError("Unsupported axle identifier.")
    if not np.array_equal(axles.Truck, np.where(axles.Axle <= 2, 1, 2)):
        raise ValueError("Truck identifiers disagree with their axle mapping.")
    truck_keys = set(trucks[TRUCK_KEY].itertuples(index=False, name=None))
    axle_keys = set(axles[TRUCK_KEY].itertuples(index=False, name=None))
    if truck_keys != axle_keys:
        raise ValueError("Truck and axle/rail key coverage must match exactly.")
    if not np.isfinite(axles[list(columns)].to_numpy(dtype=float)).all():
        raise ValueError("Own-axle predictors must be finite.")
    lookup = {}
    for key, group in axles.groupby(TRUCK_KEY, sort=False):
        expected = {1, 2} if key[1] == 1 else {3, 4}
        if len(group) != 2 or set(group.Axle) != expected:
            raise ValueError("Each truck/rail row must have its two correct axles.")
        lookup[key] = group.sort_values("Axle")[list(columns)].to_numpy(dtype=float)
    return np.stack([lookup[key] for key in trucks[TRUCK_KEY].itertuples(index=False, name=None)])


def _pair_array(values):
    array = np.asarray(values, dtype=float)
    if array.ndim != 3 or array.shape[1] != 2 or not len(array) or not np.isfinite(array).all():
        raise ValueError("Pair features must be a nonempty finite (n, 2, p) array.")
    return array


def _rail_array(values, size):
    array = np.asarray(values)
    if array.shape != (size,) or not np.isin(array, RAIL_PROFILES).all():
        raise ValueError("Each pair must have one supported rail identifier.")
    return array


def paired_kernel(left, right, left_rails, right_rails, gamma):
    """Normalized permutation-average product kernel on standardized pairs.

    For A=(a1,a2), B=(b1,b2), H(A,B) averages g(a1,b1)g(a2,b2)
    and g(a1,b2)g(a2,b1), with g=exp(-gamma*d²/18). Normalize H by
    sqrt(H(A,A)H(B,B)), then multiply the original rail kernel once.
    """
    left, right = _pair_array(left), _pair_array(right)
    if left.shape[2] != right.shape[2]:
        raise ValueError("Both sides need the same numeric features.")
    if not np.isfinite(gamma) or gamma <= 0:
        raise ValueError("gamma must be finite and positive.")
    left_rails = _rail_array(left_rails, len(left))
    right_rails = _rail_array(right_rails, len(right))

    def gaussian(a, b):
        distance = np.maximum((a * a).sum(axis=1)[:, None]
                              + (b * b).sum(axis=1)[None, :] - 2 * a @ b.T, 0.)
        return np.exp(-gamma * distance / (2 * DISTANCE_SCALE))

    h = .5 * (gaussian(left[:, 0], right[:, 0]) * gaussian(left[:, 1], right[:, 1])
              + gaussian(left[:, 0], right[:, 1]) * gaussian(left[:, 1], right[:, 0]))

    def own_diagonal(array):
        distance = np.sum((array[:, 0] - array[:, 1]) ** 2, axis=1)
        return .5 * (1. + np.exp(-gamma * distance / DISTANCE_SCALE))

    normalized = h / np.sqrt(own_diagonal(left)[:, None] * own_diagonal(right)[None, :])
    # One-hot rail vectors differ by squared distance 2, or 0 for the same rail.
    rail_distance = 2. * (left_rails[:, None] != right_rails[None, :])
    return normalized * np.exp(-gamma * rail_distance / DISTANCE_SCALE)


def pair_scores(train_features, test_features, train_rails, test_rails,
                train_target, alpha, gamma):
    """Fit centered signed-label ridge with scaling learned from fitting pairs."""
    train_features, test_features = _pair_array(train_features), _pair_array(test_features)
    if train_features.shape[2] != test_features.shape[2]:
        raise ValueError("Training and test numeric dimensions must match.")
    target = np.asarray(train_target)
    if target.shape != (len(train_features),) or not np.isin(target, [0, 1]).all():
        raise ValueError("Training labels must be a binary vector matching the pairs.")
    if not np.isfinite(alpha) or alpha <= 0:
        raise ValueError("alpha must be finite and positive.")
    mean = train_features.mean(axis=(0, 1))
    scale = train_features.std(axis=(0, 1))
    scale = np.where(scale == 0, 1., scale)
    train = (train_features - mean) / scale
    test = (test_features - mean) / scale
    signed = 2 * target - 1
    target_mean = signed.mean()
    weights = np.linalg.solve(
        paired_kernel(train, train, train_rails, train_rails, gamma)
        + alpha * np.eye(len(train)), signed - target_mean)
    return paired_kernel(test, train, test_rails, train_rails, gamma) @ weights + target_mean


def evaluate_paired_models(trucks, axles, target, feature_sets, parameter_grid=PARAMETER_GRID):
    """Nested complete-car evaluation on truck labels, preserving both axles.

    Returns aggregate metrics, wide binary predictions, selected settings and
    wide continuous signed scores. Every outer row is predicted once. Feature
    ideas remain exploratory even though parameter tuning stays inside folds.
    """
    _check_truck_keys(trucks)
    if target not in ("acceleration_hunted", "displacement_hunted"):
        raise ValueError("Choose the existing truck acceleration or displacement target.")
    if target not in trucks or not trucks[target].isin([0, 1]).all():
        raise ValueError("Truck target labels must be binary.")
    reserved = set(TRUCK_KEY + ["actual"])
    if not feature_sets or any(not isinstance(name, str) or not name or name in reserved for name in feature_sets):
        raise ValueError("Model names must preserve identifier and outcome columns.")
    grid = tuple(parameter_grid)
    if not grid or any(len(pair) != 2 or not np.isfinite(pair).all() or min(pair) <= 0 for pair in grid):
        raise ValueError("The fixed grid needs positive finite alpha/gamma pairs.")
    trucks = trucks.reset_index(drop=True)
    features = {name: make_pair_tensor(trucks, axles, columns) for name, columns in feature_sets.items()}
    car_ids = trucks.Car.to_numpy()
    cars = np.sort(np.unique(car_ids))
    if len(cars) < 4:
        raise ValueError("At least four complete car groups are required.")
    y, rails = trucks[target].to_numpy(dtype=int), trucks.Rail_Profile.to_numpy()
    if set(np.unique(y)) != {0, 1}:
        raise ValueError("Both truck outcome classes are required.")
    predictions = trucks[TRUCK_KEY].copy()
    predictions["actual"] = y
    scores = predictions.copy()
    settings = []

    def fit_scores(name, fit, test, alpha, gamma):
        if set(car_ids[fit]) & set(car_ids[test]):
            raise AssertionError("Fitting and held-out car groups overlap.")
        if set(np.unique(y[fit])) != {0, 1}:
            raise ValueError("A fitting fold has only one truck target class.")
        return pair_scores(features[name][fit], features[name][test], rails[fit], rails[test],
                           y[fit], alpha, gamma)

    for name in feature_sets:
        outer_scores = np.full(len(trucks), np.nan)
        seen = np.zeros(len(trucks), dtype=int)
        for held_car in cars:
            train = np.flatnonzero(car_ids != held_car)
            test = np.flatnonzero(car_ids == held_car)
            inner_folds = np.array_split(cars[cars != held_car], 3)
            candidates = []
            for alpha, gamma in grid:
                inner_scores = np.full(len(trucks), np.nan)
                inner_seen = np.zeros(len(trucks), dtype=int)
                for validation_cars in inner_folds:
                    validation = train[np.isin(car_ids[train], validation_cars)]
                    fit = train[~np.isin(car_ids[train], validation_cars)]
                    inner_scores[validation] = fit_scores(name, fit, validation, alpha, gamma)
                    inner_seen[validation] += 1
                if not np.all(inner_seen[train] == 1) or np.any(inner_seen[test]):
                    raise AssertionError("Inner validation must cover fitting cars exactly once.")
                if not np.isfinite(inner_scores[train]).all():
                    raise AssertionError("All inner held-out scores must be finite.")
                ba = classification_metrics(y[train], (inner_scores[train] >= 0).astype(int))["balanced_accuracy"]
                candidates.append((ba, float(alpha), float(gamma)))
            inner_ba, alpha, gamma = max(candidates, key=lambda row: (row[0], row[1], -row[2]))
            outer_scores[test] = fit_scores(name, train, test, alpha, gamma)
            seen[test] += 1
            settings.append(dict(held_car=int(held_car), model=name, alpha=alpha, gamma=gamma,
                                 inner_balanced_accuracy=inner_ba))
        if not np.all(seen == 1) or not np.isfinite(outer_scores).all():
            raise AssertionError("Every truck must receive one finite held-out score.")
        predictions[name] = (outer_scores >= 0).astype(int)
        scores[name] = outer_scores
    metrics = pd.DataFrame([dict(target=target, model=name,
                                 **classification_metrics(y, predictions[name])) for name in feature_sets])
    return metrics, predictions, pd.DataFrame(settings), scores


In [ ]:
"""One fixed paired-axle descriptor comparison; private local results only."""
from pathlib import Path
import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd

earlier_protected_state = protected_state
from contact_hunting import evaluate_models, classification_metrics, paired_car_intervals
TARGETS = ("acceleration_hunted", "displacement_hunted")


def protected_state():
    return earlier_protected_state()


def check_prior(pred, scenario, target):
    if PREVIOUS is None:
        return
    prior = pd.read_csv(PREVIOUS / f"{scenario}_{target}_predictions.csv")
    joined = pred.merge(prior, on=KEY, validate="one_to_one", suffixes=("_new", "_old"))
    assert len(joined) == len(pred) == len(prior)
    assert np.array_equal(joined.actual_new, joined.actual_old)
    for current, previous in OLD_NAMES.items():
        assert np.array_equal(joined[current], joined[previous]), (scenario, target, current)


def independently_check_metrics(row, actual, predicted):
    y, p = np.asarray(actual), np.asarray(predicted)
    tn, fp, fn, tp = (int(((y == a) & (p == b)).sum()) for a, b in ((0, 0), (0, 1), (1, 0), (1, 1)))
    assert [tn, fp, fn, tp] == [row[c] for c in ("true_negative", "false_positive", "false_negative", "true_positive")]
    assert np.isclose(row["balanced_accuracy"], .5 * (tn / (tn + fp) + tp / (tp + fn)))
    for name, direct in {"recall": tp / (tp + fn), "specificity": tn / (tn + fp),
                         "precision": tp / max(tp + fp, 1), "accuracy": (tp + tn) / len(y)}.items():
        assert np.isclose(row[name], direct)


def run_experiment(output_dir=None):
    before = protected_state()
    trucks, axles, _, _ = load_inputs()
    rows, prediction_frames, settings_frames, comparison_frames, diagnostic_rows = [], [], [], [], []
    axle_counts = axles.groupby(KEY).displacement_hunted.sum().rename("displacement_positive_axles").reset_index()
    for scenario in ("all_cars", "without_car_7"):
        t = trucks.copy() if scenario == "all_cars" else trucks[trucks.Car != 7].reset_index(drop=True)
        a = axles.copy() if scenario == "all_cars" else axles[axles.Car != 7].reset_index(drop=True)
        for target in TARGETS:
            _, original, original_settings = evaluate_models(t, target, FEATURE_SETS)
            check_prior(original, scenario, target)
            _, paired, paired_settings, scores = evaluate_paired_models(t, a, target, FEATURE_SETS)
            pd.testing.assert_frame_equal(original[KEY + ["actual"]], paired[KEY + ["actual"]])
            pd.testing.assert_frame_equal(paired[KEY + ["actual"]], scores[KEY + ["actual"]])
            comparison = original[KEY + ["actual"]].copy()
            for representation, frame, chosen in (("Original summary", original, original_settings),
                                                   ("Paired axles", paired, paired_settings)):
                chosen = chosen.copy()
                chosen["representation"], chosen["scenario"], chosen["target"] = representation, scenario, target
                settings_frames.append(chosen)
                for model in FEATURE_SETS:
                    pred = frame[model].to_numpy()
                    metric = dict(scenario=scenario, target=target, representation=representation,
                                  model=model, observations=len(frame),
                                  **classification_metrics(frame.actual, pred))
                    independently_check_metrics(metric, frame.actual, pred)
                    rows.append(metric)
                    long = frame[KEY + ["actual"]].copy()
                    long["predicted"] = pred
                    if representation == "Paired axles":
                        long["score"] = scores[model]
                        assert np.array_equal(pred, scores[model].to_numpy() >= 0)
                    long["scenario"], long["target"] = scenario, target
                    long["representation"], long["model"] = representation, model
                    prediction_frames.append(long)
                    comparison[representation + ": " + model] = pred
            contrasts = []
            for representation in ("Original summary", "Paired axles"):
                prefix = representation + ": "
                contrasts.extend([(prefix + "Conicity + shapes", prefix + "Conicity", prefix + "shape beyond conicity"),
                                  (prefix + "Wear + shapes", prefix + "Wear", prefix + "shape beyond wear")])
            contrasts.extend([("Paired axles: " + model, "Original summary: " + model,
                               "Paired versus summary: " + model) for model in FEATURE_SETS])
            _, changes = paired_car_intervals(comparison, n_bootstrap=2000, seed=42, comparisons=contrasts)
            changes["scenario"], changes["target"] = scenario, target
            comparison_frames.append(changes)
            for model in FEATURE_SETS:
                for car, g in comparison.groupby("Car"):
                    old_errors = int((g["Original summary: " + model] != g.actual).sum())
                    new_errors = int((g["Paired axles: " + model] != g.actual).sum())
                    diagnostic_rows.append(dict(kind="per_car", scenario=scenario, target=target, model=model,
                                                Car=int(car), observations=len(g), original_errors=old_errors,
                                                paired_errors=new_errors, error_reduction=old_errors-new_errors))
            z = comparison.merge(axle_counts, on=KEY, validate="one_to_one")
            context = t[KEY + ["acceleration_hunted", "displacement_hunted"]].copy()
            context["endpoint_disagreement"] = context.acceleration_hunted != context.displacement_hunted
            z = z.merge(context[KEY + ["endpoint_disagreement"]], on=KEY, validate="one_to_one")
            for representation in ("Original summary", "Paired axles"):
                for model in FEATURE_SETS:
                    column = representation + ": " + model
                    groups = [(f"displacement_positive_axles_{n}", z[z.displacement_positive_axles == n]) for n in (0, 1, 2)]
                    groups += [("endpoint_agreement", z[~z.endpoint_disagreement]),
                               ("endpoint_disagreement", z[z.endpoint_disagreement])]
                    for group, g in groups:
                        diagnostic_rows.append(dict(kind="subgroup", scenario=scenario, target=target,
                                                    representation=representation, model=model, subgroup=group,
                                                    observations=len(g), correct=int((g.actual == g[column]).sum()),
                                                    true_positive=int(((g.actual == 1) & (g[column] == 1)).sum()),
                                                    true_negative=int(((g.actual == 0) & (g[column] == 0)).sum()),
                                                    false_positive=int(((g.actual == 0) & (g[column] == 1)).sum()),
                                                    false_negative=int(((g.actual == 1) & (g[column] == 0)).sum())))
            print(f"Completed {scenario}, {target}; summary-model predictions computed", flush=True)
    assert before == protected_state(), "An original input, prior notebook or implementation changed"
    result = dict(metrics=pd.DataFrame(rows), comparisons=pd.concat(comparison_frames, ignore_index=True),
                  predictions=pd.concat(prediction_frames, ignore_index=True),
                  settings=pd.concat(settings_frames, ignore_index=True), diagnostics=pd.DataFrame(diagnostic_rows))
    assert not result["predictions"].duplicated(["scenario", "target", "representation", "model", *KEY]).any()
    verification = dict(protected_state=before, originals_unchanged=True,
                        historical_predictions_checked=PREVIOUS is not None,
                        python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                        feature_sets=FEATURE_SETS, bootstrap_draws=2000, seed=42,
                        kernel="Unit-diagonal normalized average of both axle product matchings; numeric distances averaged; rail once",
                        stopping_rule="One structural comparison; no further representation/grid/threshold search after results",
                        note="Repeated exploratory sample of 14 cars; uncalibrated scores; not full wheel curves or causal/localization evidence")
    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        for name, frame in result.items():
            frame.to_csv(output_dir / f"{name}.csv", index=False)
        (output_dir / "verification.json").write_text(json.dumps(verification, indent=2) + "\n")
    result["verification"] = verification
    return result


## Comparison design

The question is whether combinations of measurements **belonging to each actual axle** matter. A truck mean can hide that one axle has high conicity and low nonlinearity while the other has the reverse combination. The new model keeps these feature correspondences within each axle and considers both axle matchings when comparing trucks. It does not predict which axle is unstable.

| Held fixed | Both representations |
|---|---|
| Observation unit and labels | Same 196 truck/rail assessments; acceleration primary, displacement sensitivity |
| Independent groups | Same 14 measured cars; complete cars held out |
| Classifier and search | Centered signed-target kernel ridge, zero threshold, same six alpha/gamma settings |
| Feature bundles | Conicity; conicity + three shapes; conicity + four wear means; conicity + wear + shapes |
| Evaluation | Three inner car folds select pooled balanced accuracy; outer car holdout estimates predictions |

Every bundle includes categorical rail. “Wear” also includes conicity and rail. Own-axle conicity and two-wheel wear/shape means enter the pair; the original model uses truck means and maximum left/right distance. For axle data, the inherited column `conicity_mean` is own-axle conicity and `max_left_right_shape_difference` is own-axle RMS distance with no maximum across axles.

We use training-only scaling pooled over both axle positions, average both matchings, normalize to unit diagonal and apply rail weighting once. We evaluate both outcomes and check errors for all cars. This is one structural experiment; no new thresholds, features or kernel alternatives will be searched afterward.

**Limits:** this changes representation and kernel geometry together, so it cannot isolate averaging alone. Axle exchangeability is an assumption; front/rear effects are not tested. Coordinate units, orientation and reconstruction remain provisional. These same 14 cars have been explored repeatedly.

### Exact construction

For standardized axle vectors, $g(a,b)=\exp[-\gamma\|a-b\|^2/18]$. For trucks $A=(a_1,a_2)$ and $B=(b_1,b_2)$,

$$H(A,B)=\frac{g(a_1,b_1)g(a_2,b_2)+g(a_1,b_2)g(a_2,b_1)}{2}.$$

Normalize by $\sqrt{H(A,A)H(B,B)}$ and multiply once by the existing rail kernel $\exp[-\gamma\|r_A-r_B\|^2/9]$. Numeric distances are averaged over the two axles; rail contributes once. Mean and population SD are fitted on pooled axle values from fitting trucks only. The construction is symmetric, positive semidefinite and unchanged by swapping either axle pair. If both axles of every truck are identical, it reduces to the original Gaussian model.

The construction is grounded in [permutation-average product kernels](https://ijcai.org/proceedings/2007/117). This mathematical validity does not imply railway physics or an expected score gain. Scores remain uncalibrated decisions, not probabilities.

In [ ]:
# Fresh fixed evaluation; results stay in outputs/.
result=run_experiment(output_dir=folder)
metrics=result['metrics'];comparisons=result['comparisons']
predictions=result['predictions'];diagnostics=result['diagnostics']
trucks,axles,wheels,profiles=load_inputs()
display(pd.Series({'Cars':trucks.Car.nunique(),'Trucks':len(trucks[['Car','Truck']].drop_duplicates()),
                   'Truck/rail assessments':len(trucks),'Wheel profiles':len(wheels)},name='Coverage'))
# First mixed-outcome case, by declared lexicographic key; illustration, not a selected success.
counts=axles.groupby(KEY).displacement_hunted.sum()
example_key=counts[counts==1].sort_index().index[0]
example=axles.set_index(KEY).loc[[example_key]].reset_index()
display(example[KEY+['Axle',*FEATURE_SETS['Wear + shapes']]].round(4))


## Full comparison on held-out cars

Balanced accuracy averages hunting and no-hunting recall. The two endpoints are kept separate, and every planned bundle is shown. The same complete-car outer folds are used; scaling and parameter choice stay inside training cars. We recompute the original summary-model decisions for both endpoints and both samples. Set HUNTING_REFERENCE_DIR only if the private historical-reference CSVs are available; the check is optional and the output verification file records whether it ran.

In [ ]:
main=metrics[metrics.scenario=='all_cars'].copy()
rename={'balanced_accuracy':'Balanced accuracy (%)','recall':'Hunting recall (%)','specificity':'No-hunting recall (%)'}
display((100*main.set_index(['target','model','representation'])[['balanced_accuracy','recall','specificity']]).round(1).rename(columns=rename))
fig,axes=plt.subplots(2,3,figsize=(15,8),sharey=True)
x=np.arange(len(FEATURE_SETS))
for row,(target,label) in enumerate(zip(TARGETS,['Acceleration (primary)','Displacement (sensitivity)'])):
    for col,(metric,title) in enumerate(zip(['balanced_accuracy','recall','specificity'],['Balanced accuracy','Hunting recall','No-hunting recall'])):
        axis=axes[row,col]
        for offset,rep,color in [(-.19,'Original summary','#7e8c98'),(.19,'Paired axles','#226584')]:
            s=main[(main.target==target)&(main.representation==rep)].set_index('model').loc[list(FEATURE_SETS),metric]
            axis.bar(x+offset,100*s,width=.36,label=rep,color=color)
        axis.set_xticks(x,['Conicity','Conicity+shape','Wear','Wear+shape'])
        axis.tick_params(axis='x',labelsize=9);axis.set(title=title,ylim=(0,100))
        axis.grid(axis='y',alpha=.18);axis.set_axisbelow(True)
    axes[row,0].set_ylabel(label+' (%)')
axes[0,0].legend(fontsize=9);fig.suptitle('Direct truck labels in both models | 196 assessments from 14 source cars')
fig.tight_layout();fig.savefig(folder / 'paired_review_performance.png', dpi=300); fig.savefig(folder / 'paired_review_performance.pdf', bbox_inches='tight');plt.show()


## Do shape and pairing add useful information?

Ranges come from 2,000 paired resamples of complete-car blocks of fixed outer predictions, seed 42. They omit retraining and feature-discovery uncertainty and are unadjusted for the multiple planned comparisons. A range crossing zero leaves a gain uncertain. A comparison using the original summary model still uses the same source cars and does not provide independent new-car validation.

In [ ]:
main_changes=comparisons[comparisons.scenario=='all_cars']
view=main_changes[['target','comparison','change_in_balanced_accuracy','lower_95','upper_95',
                   'cars_with_fewer_errors','cars_with_more_errors','cars_tied']].copy()
for c in ['change_in_balanced_accuracy','lower_95','upper_95']:view[c]*=100
display(view.round(1).rename(columns={'change_in_balanced_accuracy':'Gain (pp)','lower_95':'Lower (pp)','upper_95':'Upper (pp)'}))
short=['Summary: shape beyond conicity','Summary: shape beyond wear','Pair: shape beyond conicity','Pair: shape beyond wear',
       'Pair versus summary: conicity','Pair versus summary: conicity + shape','Pair versus summary: wear','Pair versus summary: wear + shape']
fig,axes=plt.subplots(1,2,figsize=(14,5.6),sharey=True)
for axis,target,title in zip(axes,TARGETS,['Acceleration (primary)','Displacement (sensitivity)']):
    q=main_changes[main_changes.target==target];y=np.arange(len(q))
    axis.hlines(y,100*q.lower_95,100*q.upper_95,color='#226584',linewidth=2)
    axis.scatter(100*q.change_in_balanced_accuracy,y,color='#226584',zorder=3)
    axis.axvline(0,color='#777777',linewidth=.9);axis.set_yticks(y,short)
    axis.set(title=title,xlabel='Balanced-accuracy change (percentage points)');axis.grid(axis='x',alpha=.15)
axes[0].invert_yaxis();fig.suptitle('Paired complete-car descriptive ranges | fixed predictions')
fig.tight_layout();fig.savefig(folder / 'paired_review_changes.png', dpi=300); fig.savefig(folder / 'paired_review_changes.pdf', bbox_inches='tight');plt.show()


## All-car checks, not a few favorable examples

The heatmap shows errors removed by changing representation, for every car and feature bundle. Positive means fewer errors; negative means more. These are error counts, not car-specific balanced accuracies. Shared source cars and repeated rails remain visible.

The mixed-axle and endpoint-disagreement tables are descriptive. A direct truck model has no axle-localization output, so a correct truck decision cannot establish that it identified the unstable axle.

In [ ]:
car=diagnostics[(diagnostics.kind=='per_car')&(diagnostics.scenario=='all_cars')]
display(car.pivot(index=['target','Car'],columns='model',values='error_reduction').astype(int))
fig,axes=plt.subplots(1,2,figsize=(14,4.3),sharey=True)
limit=max(1,int(car.error_reduction.abs().max()))
for axis,target,title in zip(axes,TARGETS,['Acceleration','Displacement']):
    q=car[car.target==target].pivot(index='model',columns='Car',values='error_reduction').loc[list(FEATURE_SETS)]
    im=axis.imshow(q.to_numpy(),cmap='RdBu',vmin=-limit,vmax=limit,aspect='auto')
    axis.set_xticks(range(len(q.columns)),[str(int(c)) for c in q.columns]);axis.set_yticks(range(len(q)),q.index)
    for i in range(q.shape[0]):
        for j in range(q.shape[1]):axis.text(j,i,str(int(q.iloc[i,j])),ha='center',va='center',fontsize=9,
                                           color='white' if abs(q.iloc[i,j])>.6*limit else '#222222')
    axis.set(title=title,xlabel='Held-out car')
fig.subplots_adjust(left=.13,right=.86,bottom=.17,top=.85,wspace=.18)
cax=fig.add_axes([.89,.19,.015,.6])
fig.colorbar(im,cax=cax,label='Original errors minus paired errors')
fig.suptitle('Every held-out car | blue: fewer errors; red: more errors')
fig.savefig(folder / 'paired_review_cars.png', dpi=300); fig.savefig(folder / 'paired_review_cars.pdf', bbox_inches='tight');plt.show()
sub=diagnostics[(diagnostics.kind=='subgroup')&(diagnostics.scenario=='all_cars')].copy()
sub['Correct (%)']=100*sub.correct/sub.observations
# All bundles/endpoints are retained, including unfavorable subgroup results.
display(sub[['target','representation','model','subgroup','observations','correct','false_positive','false_negative','Correct (%)']].round(1))
# Every one-positive-axle case: truck decisions only; no axle-localization claim.
mixed_keys=counts[counts==1].rename('Positive axles').reset_index()[KEY]
mixed=predictions[(predictions.scenario=='all_cars') & (predictions.target=='displacement_hunted')].merge(
    mixed_keys,on=KEY,validate='many_to_one')
display(mixed.pivot(index=KEY,columns=['representation','model'],values='predicted'))



## Car 7 sensitivity

Car 7’s source anomaly is retained in the main experiment. This prespecified sensitivity removes the car entirely and reruns training, tuning and evaluation for both representations. It changes the sample; it is not a confidence interval and does not replace the all-car result.

In [ ]:
sensitivity=metrics[metrics.scenario=='without_car_7']
display((100*sensitivity.set_index(['target','model','representation'])[['balanced_accuracy','recall','specificity']]).round(1).rename(columns=rename))
s=comparisons[comparisons.scenario=='without_car_7'].copy()
for c in ['change_in_balanced_accuracy','lower_95','upper_95']:s[c]*=100
display(s[['target','comparison','change_in_balanced_accuracy','lower_95','upper_95']].round(1)
        .rename(columns={'change_in_balanced_accuracy':'Gain (pp)','lower_95':'Lower (pp)','upper_95':'Upper (pp)'}))


## Interpreting the comparison

We compare the complete feature bundles using balanced accuracy, hunting recall and no-hunting recall together. The paired ranges and per-car checks show whether a change is consistent across the sample or concentrated in a few cars. The standard-wear comparison tests whether the three WHL descriptors add information beyond ordinary measurements.

For the axle-OR experiment, we also inspect false sister-axle flags and whether a correct truck prediction identified the positive axle. For the paired experiment, the model predicts the truck outcome and cannot identify an unstable axle. These are different modeling questions.

We keep favorable and unfavorable results in the generated tables. The same cars have informed repeated feature development, so the results remain exploratory. Coordinate units, orientation and reconstruction also limit physical interpretation. We need new cars to test transfer and controlled profile changes to investigate effects within the simulator. The scores do not establish causes or maintenance thresholds.


In [ ]:
assert result['verification']['protected_state']==protected_state()
assert not predictions.duplicated(['scenario','target','representation','model',*KEY]).any()
for row in metrics.itertuples():
    p=predictions[(predictions.scenario==row.scenario)&(predictions.target==row.target)&
                  (predictions.representation==row.representation)&(predictions.model==row.model)]
    direct=.5*sum((p.loc[p.actual==k,'predicted']==k).mean() for k in [0,1])
    assert np.isclose(direct,row.balanced_accuracy)
print('Verified: summary comparators recomputed; metrics independently checked; inputs and code unchanged.')
